# 03 · Baseline (MLP) analysis

Uses the MLP run `e1_mlp_xyz/seed42` (`make train-baseline`). If it does not exist yet, the 5-epoch smoke run is shown and explicitly labelled - smoke numbers are **not** results.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from skeleton_action.constants import CLASS_NAMES, JOINT_NAMES, KARD_EDGES, J
from skeleton_action.data.kard import KardCollection
INTERIM = ROOT / "data/interim/kard_realworld.npz"
assert INTERIM.exists(), "run `make data` first"
col = KardCollection.load(INTERIM)
print(len(col), "sequences")

In [ ]:
def load_run(name, seed):
    m = ROOT / "outputs/metrics" / name / f"seed{seed}"
    if not (m / "summary.json").exists():
        return None
    out = {"summary": json.loads((m / "summary.json").read_text()),
           "history": pd.read_csv(m / "history.csv")}
    for part in ("val", "test"):
        p = m / f"{part}_metrics.json"
        if p.exists():
            out[part] = json.loads(p.read_text())
    return out
run, label = load_run("e1_mlp_xyz", 42), "e1_mlp_xyz/seed42"
if run is None:
    run, label = load_run("smoke_mlp", 0), "SMOKE RUN (5 epochs, pipeline check only)"
print(label if run else "no MLP run found: run `make train-baseline`")

In [ ]:
if run:
    h = run["history"]
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    ax[0].plot(h.epoch, h.train_loss, label="train"); ax[0].plot(h.epoch, h.val_loss, label="val"); ax[0].set_title("loss")
    ax[1].plot(h.epoch, h.train_macro_f1, label="train"); ax[1].plot(h.epoch, h.val_macro_f1, label="val"); ax[1].set_title("macro-F1")
    for a in ax: a.legend()
    fig.suptitle(label); plt.tight_layout(); plt.show()

In [ ]:
if run and "test" in run:
    t = run["test"]
    print(label, "| test subjects", t["subjects"], "| acc %.3f macro-F1 %.3f" % (t["accuracy"], t["macro_f1"]))
    pc = pd.DataFrame(t["per_class"]).T.sort_values("f1")
    pc[["precision", "recall", "f1"]].plot.barh(figsize=(7, 7), title="per-class (test)"); plt.show()
    display(pd.DataFrame(t["top_confusions"]))

### Why temporal order matters for the MLP
`aggregation: segments` keeps coarse order; an order-invariant mean pooling cannot separate time-reversed classes such as Sit down / Stand up without velocity input (see `tests/test_models.py`).